# LexChain parser benchmark — Docling vs MinerU vs Marker on OHR-Bench (Law)

**Kaggle setup:** GPU T4 x2 accelerator, internet ON.

Flow: **Cell 1** setup → **Cell 2** 3-document smoke test + full-run time estimate → *approve* → **Cell 3** full run (95 docs) → **Cell 4** metrics table + CSV.

**If the session dies:** progress is checkpointed after every document in `/kaggle/working/results`. In a new session, add the previous run's output as an *Input dataset*, then pass `--restore-from /kaggle/input/<slug>/results` in Cell 3.

In [ ]:
# Cell 1 — clone repo + set up envs, data, models (~15-25 min first time)
REPO_URL = "https://github.com/YOUR_USERNAME/lexchain-parser-bench.git"  # <-- set me

import os
if not os.path.exists("/kaggle/working/lexchain-parser-bench"):
    !git clone {REPO_URL} /kaggle/working/lexchain-parser-bench
%cd /kaggle/working/lexchain-parser-bench
!git pull
!nvidia-smi -L
!bash setup.sh

In [ ]:
# Cell 2 — smoke test: 3 smallest PDFs through all 3 tools, then estimate the full run
!python run_benchmark.py --limit 3
!python evaluate.py

import json
from pathlib import Path

results = Path("/kaggle/working/results")
gt_dir = Path("/kaggle/tmp/ohr_data/gt/law")
total_pages = sum(len(json.loads(p.read_text())) for p in gt_dir.glob("*.json"))
print(f"\n=== Full-run estimate ({total_pages} pages, {len(list(gt_dir.glob('*.json')))} docs) ===")
rates, loads = {}, {}
for tool in ["docling", "mineru", "marker"]:
    metas = [json.loads(p.read_text()) for p in (results / tool / "meta").glob("*.json")]
    ok = [m for m in metas if m.get("status") == "success" and m.get("wall_s") and m.get("pages")]
    if not ok:
        print(f"{tool}: NO successful smoke docs — check results/{tool}/worker.log before the full run")
        continue
    rates[tool] = sum(m["wall_s"] for m in ok) / sum(m["pages"] for m in ok)
    lt = results / tool / "load_time.json"
    loads[tool] = json.loads(lt.read_text())["model_load_s"] if lt.exists() else 0
    print(f"{tool}: {rates[tool]:.1f} s/page (model load {loads[tool]:.0f}s) "
          f"-> ~{(rates[tool] * total_pages + loads[tool]) / 3600:.1f} h alone")
if len(rates) == 3:
    t = sorted((rates[k] * total_pages + loads[k] for k in rates), reverse=True)
    makespan = max(t[0], t[1] + t[2])
    print(f"\nProjected wall time on 2 GPUs: ~{makespan / 3600:.1f} h")
    print("Kaggle GPU sessions cap at ~12 h. If the estimate exceeds that, run in chunks —\n"
          "every document is checkpointed, rerunning Cell 3 resumes where it left off.")

**STOP — review the estimate above before running the full benchmark.**

Resuming a previous session? Attach its output as an input dataset and use:
`!python run_benchmark.py --restore-from /kaggle/input/<slug>/results`

In [ ]:
# Cell 3 — full run (resumable; smoke-test docs are skipped automatically)
!python run_benchmark.py

In [ ]:
# Cell 4 — evaluate + paper table (CSV + markdown saved to /kaggle/working/results)
!python evaluate.py

import pandas as pd
from IPython.display import display
display(pd.read_csv("/kaggle/working/results/results_summary.csv"))
print(open("/kaggle/working/results/results.md").read())